In [ ]:
# CELL 1 - Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive connected!")

In [ ]:
# CELL 2 - Install Libraries & Download AI Model
!pip install mediapipe==0.10.14 opencv-python pandas requests > /dev/null 2>&1

import mediapipe as mp
import cv2
import pandas as pd
import os
import requests
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

print("✅ Libraries ready!")

# Download the Heavy Model
model_path = '/content/drive/MyDrive/Dissertation Project/pose_landmarker_heavy.task'

if not os.path.exists(model_path):
    print("Downloading AI model... (this takes 1-2 minutes)")
    url = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task"
    r = requests.get(url)
    with open(model_path, 'wb') as f:
        f.write(r.content)
    print("✅ AI Model downloaded and saved!")
else:
    print("✅ AI Model already exists, skipping download!")

In [ ]:
# CELL 3 - Download All Video Clips
!pip install yt-dlp > /dev/null 2>&1
import subprocess
import os

base_path = '/content/drive/MyDrive/Dissertation Project'

clips = [
    # Video 1
    ("https://youtu.be/t28jXbF5rL8", "5:31", "off_break", "v1_ob1"),
    ("https://youtu.be/t28jXbF5rL8", "6:00", "off_break", "v1_ob2"),
    ("https://youtu.be/t28jXbF5rL8", "6:07", "off_break", "v1_ob3"),
    ("https://youtu.be/t28jXbF5rL8", "7:04", "off_break", "v1_ob4"),

    # Video 2
    ("https://youtu.be/iJ2FxCGiVDM", "0:57", "carrom_ball", "v2_cb1"),
    ("https://youtu.be/iJ2FxCGiVDM", "1:25", "carrom_ball", "v2_cb2"),
    ("https://youtu.be/iJ2FxCGiVDM", "3:13", "carrom_ball", "v2_cb3"),
    ("https://youtu.be/iJ2FxCGiVDM", "9:19", "carrom_ball", "v2_cb4"),
    ("https://youtu.be/iJ2FxCGiVDM", "3:47", "off_break", "v2_ob1"),
    ("https://youtu.be/iJ2FxCGiVDM", "4:21", "off_break", "v2_ob2"),
    ("https://youtu.be/iJ2FxCGiVDM", "5:27", "off_break", "v2_ob3"),
    ("https://youtu.be/iJ2FxCGiVDM", "5:45", "off_break", "v2_ob4"),
    ("https://youtu.be/iJ2FxCGiVDM", "6:56", "off_break", "v2_ob5"),

    # Video 3
    ("https://youtu.be/tg7su0_R9Zw", "0:31", "off_break", "v3_ob1"),
    ("https://youtu.be/tg7su0_R9Zw", "1:36", "off_break", "v3_ob2"),
    ("https://youtu.be/tg7su0_R9Zw", "1:30", "carrom_ball", "v3_cb1"),

    # Video 4
    ("https://youtu.be/FarO1pZFLaI", "1:08", "off_break", "v4_ob1"),
    ("https://youtu.be/FarO1pZFLaI", "1:45", "off_break", "v4_ob2"),
    ("https://youtu.be/FarO1pZFLaI", "2:09", "carrom_ball", "v4_cb1"),

    # Video 5
    ("https://youtu.be/QNHBTWbGoUk", "0:41", "off_break", "v5_ob1"),
    ("https://youtu.be/QNHBTWbGoUk", "1:26", "off_break", "v5_ob2"),
    ("https://youtu.be/QNHBTWbGoUk", "1:50", "off_break", "v5_ob3"),
    ("https://youtu.be/QNHBTWbGoUk", "3:37", "off_break", "v5_ob4"),
    ("https://youtu.be/QNHBTWbGoUk", "4:45", "off_break", "v5_ob5"),
    ("https://youtu.be/QNHBTWbGoUk", "3:29", "carrom_ball", "v5_cb1"),
    ("https://youtu.be/QNHBTWbGoUk", "5:43", "carrom_ball", "v5_cb2"),
    ("https://youtu.be/QNHBTWbGoUk", "6:16", "carrom_ball", "v5_cb3"),
]

print(f"🏏 Starting download of {len(clips)} clips...")
success = 0
failed = 0

for url, timestamp, category, name in clips:
    output_path = os.path.join(base_path, category, f"{name}.mp4")

    # Convert timestamp to seconds
    parts = timestamp.split(":")
    seconds = int(parts[0]) * 60 + int(parts[1])
    end_seconds = seconds + 3

    cmd = [
        "yt-dlp",
        "--download-sections", f"*{seconds}-{end_seconds}",
        "--force-keyframes-at-cuts",
        "-f", "mp4",
        "-o", output_path,
        url
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode == 0:
        print(f"✅ Downloaded: {name} ({category})")
        success += 1
    else:
        print(f"❌ Failed: {name} - trying alternative...")
        failed += 1

print(f"\n🏁 Done! {success} successful, {failed} failed")

In [ ]:
# CELL 4 - Batch Processor (Extract Data from All Videos)
base_path = '/content/drive/MyDrive/Dissertation Project'
model_path = '/content/drive/MyDrive/Dissertation Project/pose_landmarker_heavy.task'

# Setup the detector
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.PoseLandmarkerOptions(base_options=base_options, output_segmentation_masks=False)
detector = vision.PoseLandmarker.create_from_options(options)

# The main loop
categories = ['off_break', 'carrom_ball']
master_tracking_data = []

print("🏏 Starting batch processor...")

for category in categories:
    folder_path = os.path.join(base_path, category)

    if not os.path.exists(folder_path):
        print(f"⚠️ Folder not found: {folder_path}")
        continue

    for filename in os.listdir(folder_path):
        if filename.endswith(".mp4"):
            video_path = os.path.join(folder_path, filename)
            print(f"Processing: {filename}...")

            cap = cv2.VideoCapture(video_path)
            frame_count = 0

            while cap.isOpened():
                success, frame = cap.read()
                if not success:
                    break

                image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
                detection_result = detector.detect(mp_image)

                if detection_result.pose_landmarks:
                    landmarks = detection_result.pose_landmarks[0]
                    right_wrist = landmarks[16]
                    right_elbow = landmarks[14]

                    master_tracking_data.append({
                        'Video_Name': filename,
                        'Delivery_Type': 'Off_Break' if category == 'off_break' else 'Carrom_Ball',
                        'Frame': frame_count,
                        'Wrist_X': right_wrist.x,
                        'Wrist_Y': right_wrist.y,
                        'Elbow_X': right_elbow.x,
                        'Elbow_Y': right_elbow.y
                    })
                frame_count += 1
            cap.release()
            print(f"✅ Done: {filename}")

# Save the CSV
if master_tracking_data:
    df = pd.DataFrame(master_tracking_data)
    csv_path = os.path.join(base_path, 'Master_Ashwin_Dataset.csv')
    df.to_csv(csv_path, index=False)
    print(f"\n✅ SUCCESS! CSV saved to your Drive!")
    print(f"Total rows of data: {len(master_tracking_data)}")
else:
    print("\n❌ No data extracted. Check your folders and videos.")

In [ ]:
# CELL 5 - Build and Train the LSTM Model (Fixed Version)
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns

print("🏏 Loading dataset...")

# Load the CSV
csv_path = '/content/drive/MyDrive/Dissertation Project/Master_Ashwin_Dataset.csv'
df = pd.read_csv(csv_path)

print(f"Total rows: {len(df)}")
print(f"Delivery types: {df['Delivery_Type'].value_counts().to_dict()}")

# ── 1. BUILD SEQUENCES ──────────────────────────────────────────────
SEQUENCE_LENGTH = 20

def build_sequences(df, seq_len=SEQUENCE_LENGTH):
    sequences = []
    labels = []

    for video_name, group in df.groupby('Video_Name'):
        group = group.sort_values('Frame').reset_index(drop=True)
        features = group[['Wrist_X', 'Wrist_Y', 'Elbow_X', 'Elbow_Y']].values
        label = group['Delivery_Type'].iloc[0]

        for i in range(0, len(features) - seq_len + 1, seq_len // 2):
            seq = features[i:i + seq_len]
            if len(seq) == seq_len:
                sequences.append(seq)
                labels.append(label)

    return np.array(sequences), np.array(labels)

X, y = build_sequences(df)
print(f"\nSequences created: {len(X)}")
print(f"Sequence shape: {X.shape}")

# ── 2. ENCODE LABELS ────────────────────────────────────────────────
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"\nClasses: {le.classes_}")
print(f"Class distribution: {np.bincount(y_encoded)}")

# ── 3. TRAIN/TEST SPLIT ─────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"\nTraining sequences: {len(X_train)}")
print(f"Testing sequences: {len(X_test)}")

# ── 4. CLASS WEIGHTS (fixes the imbalance problem) ──────────────────
class_counts = np.bincount(y_train)
total = len(y_train)
class_weights = torch.FloatTensor([total / (len(class_counts) * c) for c in class_counts])
print(f"\nClass weights applied: {class_weights}")

# ── 5. PYTORCH DATASET ──────────────────────────────────────────────
class CricketDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = CricketDataset(X_train, y_train)
test_dataset = CricketDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# ── 6. LSTM MODEL ───────────────────────────────────────────────────
class CricketLSTM(nn.Module):
    def __init__(self, input_size=4, hidden_size=64, num_layers=2, num_classes=2):
        super(CricketLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                           batch_first=True, dropout=0.3)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = self.fc(lstm_out[:, -1, :])
        return out

model = CricketLSTM()
print(f"\n🧠 Model created!")

# ── 7. TRAIN ────────────────────────────────────────────────────────
# Class weights applied to loss function - this is the key fix
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

EPOCHS = 60
train_losses = []
train_accuracies = []

print(f"\n🏋️ Training for {EPOCHS} epochs...")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == y_batch).sum().item()
        total += y_batch.size(0)

    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    accuracy = correct / total * 100
    train_losses.append(avg_loss)
    train_accuracies.append(accuracy)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} — Loss: {avg_loss:.4f} — Accuracy: {accuracy:.1f}%")

print("\n✅ Training complete!")

# ── 8. EVALUATE ─────────────────────────────────────────────────────
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.numpy())
        all_labels.extend(y_batch.numpy())

test_accuracy = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels) * 100
print(f"\n🎯 Test Accuracy: {test_accuracy:.1f}%")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds,
      target_names=le.classes_, zero_division=0))

# ── 9. GRAPHS ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Training Loss
axes[0].plot(train_losses, color='red', linewidth=2)
axes[0].set_title('Training Loss Over Time', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True)

# Training Accuracy
axes[1].plot(train_accuracies, color='green', linewidth=2)
axes[1].set_title('Training Accuracy Over Time', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].grid(True)

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_,
            yticklabels=le.classes_,
            ax=axes[2])
axes[2].set_title('Confusion Matrix', fontsize=14)
axes[2].set_ylabel('Actual')
axes[2].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Dissertation Project/results_graphs.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Graphs saved to Drive!")

# Save model
torch.save(model.state_dict(),
           '/content/drive/MyDrive/Dissertation Project/ashwin_lstm_model.pth')
print("✅ Model saved to Drive!")

In [ ]:
# CELL 6 - Prediction Demo
import torch
import torch.nn as nn
import mediapipe as mp
import cv2
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# ── Load the model ──────────────────────────────────────────────────
class CricketLSTM(nn.Module):
    def __init__(self, input_size=4, hidden_size=64, num_layers=2, num_classes=2):
        super(CricketLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                           batch_first=True, dropout=0.3)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, num_classes)
        )
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = self.fc(lstm_out[:, -1, :])
        return out

model = CricketLSTM()
model.load_state_dict(torch.load(
    '/content/drive/MyDrive/Dissertation Project/ashwin_lstm_model.pth'))
model.eval()
print("✅ Model loaded!")

# ── Setup MediaPipe ─────────────────────────────────────────────────
base_options = python.BaseOptions(
    model_asset_path='/content/drive/MyDrive/Dissertation Project/pose_landmarker_heavy.task')
options = vision.PoseLandmarkerOptions(
    base_options=base_options, output_segmentation_masks=False)
detector = vision.PoseLandmarker.create_from_options(options)
print("✅ MediaPipe loaded!")

# ── Prediction function ─────────────────────────────────────────────
SEQUENCE_LENGTH = 20
class_names = ['Carrom_Ball', 'Off_Break']

def predict_delivery(video_path):
    cap = cv2.VideoCapture(video_path)
    coordinates = []

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
        result = detector.detect(mp_image)

        if result.pose_landmarks:
            landmarks = result.pose_landmarks[0]
            right_wrist = landmarks[16]
            right_elbow = landmarks[14]
            coordinates.append([
                right_wrist.x, right_wrist.y,
                right_elbow.x, right_elbow.y
            ])
    cap.release()

    if len(coordinates) < SEQUENCE_LENGTH:
        print("❌ Video too short to analyse!")
        return

    # Take middle sequence
    mid = len(coordinates) // 2
    sequence = coordinates[mid:mid + SEQUENCE_LENGTH]
    if len(sequence) < SEQUENCE_LENGTH:
        sequence = coordinates[:SEQUENCE_LENGTH]

    # Run through model
    x = torch.FloatTensor([sequence])
    with torch.no_grad():
        output = model(x)
        probabilities = torch.softmax(output, dim=1)[0]
        predicted_class = torch.argmax(probabilities).item()
        confidence = probabilities[predicted_class].item() * 100

    print("\n" + "="*40)
    print(f"🏏 DELIVERY PREDICTION RESULT")
    print("="*40)
    print(f"Prediction  : {class_names[predicted_class]}")
    print(f"Confidence  : {confidence:.1f}%")
    print(f"Off Break   : {probabilities[1].item()*100:.1f}%")
    print(f"Carrom Ball : {probabilities[0].item()*100:.1f}%")
    print("="*40)

# ── Test it on a video ──────────────────────────────────────────────
# Change this to any video in your Drive!
test_video = '/content/drive/MyDrive/Dissertation Project/off_break/v3_ob2.mp4'
predict_delivery(test_video)